# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [1]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-02-22T19:58:32.153135",
    "last_interaction": "2026-02-22T19:58:32.153180",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-02-22T19:58:32.025279",
    "last_interaction": "2026-02-22T19:58:32.025614",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:cb3cc99e-19a3-4f12-897d-5d6f4ed670ca",
    "dctIssued": "2026-02-22T19:58:32.194927Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [2]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:18a2

## Provider creates initial offer (Provider -> Consumer)

In [3]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6"
    },
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:16b3fc06-4cbd-4d93-b1

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [4]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6"
    },
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:18a2788a-3c6d-478a-b6a6-8cc5ab58e155",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [5]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:56ea51d9-7a10-4596-a404-3bf6a3a0650c",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6"
    },
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:16b3fc06-4cbd-4d93-b189-0af85e9b3a97",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [6]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:18a2788a-3c6d-478a-b6a6-8cc5ab58e155",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:57:22.077288Z",
    "updatedAt": "2026-02-22T20:57:22.421419Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:f

## Provider creates the Agreement (Provider -> Consumer)

In [7]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:16b3fc06-4cbd-4d93-b189-0af85e9b3a97",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:57:21.898174Z",
    "updatedAt": "2026-02-22T20:57:22.484833Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:f7919

## Consumer verifies the agreement (Consumer -> Provider)

In [8]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:18a2788a-3c6d-478a-b6a6-8cc5ab58e155",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:57:22.077288Z",
    "updatedAt": "2026-02-22T20:57:22.537148Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid:f

## Provider finalizes the negotiation (Provider -> Consumer)

In [9]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:f79197ac-5f9b-4cfb-95a5-00a4ff61b9e3",
    "providerPid": "urn:provider-pid:c1c68c6e-25be-48b5-a16a-f50acc9103dc",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:16b3fc06-4cbd-4d93-b189-0af85e9b3a97",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:57:21.898174Z",
    "updatedAt": "2026-02-22T20:57:22.602337Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid

## Final agreement

In [10]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
  "negotiationAgentProcessId": "urn:negotiation-process:16b3fc06-4cbd-4d93-b189-0af85e9b3a97",
  "negotiationAgentMessageId": "urn:negotiation-message:b1284066-b55a-4f9e-9f5c-6aa1f170c322",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6",
    "timestamp": "1771793842"
  },
  "target": "urn:dataset:8919d18a-e9cd-4ebc-8836-fef4cda127d6",
  "state": "ACTIVE",
  "createdAt": "2026-02-22T20:57:22.490792Z",
  "updatedAt": "2026-02-22T20:57:22.608825Z"
}

Final agreement id: 
urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [11]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pushing",
    "dataAddress": {
        "@type": "DataAddress",
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "@type": "EndpointProperty",
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "@type": "EndpointProperty",
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
    "format": "http+pushing",
    "dataAddress": {
      "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
      "endpoint": "http://127.0.0.1:1112",
      "endpointProperties": [
        {
          "name": "authorization",
          "value": "TOKEN-ABCDEFG"
        },
        {
          "name": "authType",
          "value": "bearer"
        }
      ]
    },
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:be361e0

## Start transfer (Provider -> Consumer)

In [12]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:32a995d7-8085-4f1e-803a-51a94b79a675",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f8fb0f9c-fd91-4fed-aa94-b66263bd779c",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+pushing",
    "agreementId": "urn:agreement:4951

## Suspend transfer (Consumer -> Provider)

In [13]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:be361e0a-8ccf-4f89-93c9-8dd7f8f45659",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+pushing",
    "agreementId": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",

## Restart transfer (Consumer -> Provider)

In [14]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1200/dataplane/proxy/urn:dataplane-transfer:d7cd1a05-0d87-4235-ac53-fccbd8600c23",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:be361e0a-8ccf-4f89-93c9-8dd7f8f45659",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "transferDirection": "http+pushing",
    "agreementId": "urn:agreement:495

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [15]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f8fb0f9c-fd91-4fed-aa94-b66263bd779c",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+pushing",
    "agreementId": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",

## Failure Test: Attempt start with invalid parameters

In [16]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "dataAddress": None
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": null,
  "providerPid": null,
  "code": "3120",
  "reason": [
    "Failed to deserialize the JSON body into the target type: dataAddress: unknown field `dataAddress`, expected `consumerPid` or `providerPid` at line 1 column 158",
    "Invalid Format"
  ]
}


## Failure Test: Attempt duplicate or invalid suspension

In [17]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "6030",
    "reason": [
      "TransferProcessMessageType TransferSuspensionMessage is not allowed here. Current state is SUSPENDED",
      "Failed to parse file"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [18]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:4881af95-2f42-4375-8981-8ab32bcec3d0",
    "providerPid": "urn:provider-pid:a035348a-fbaf-46d7-b627-8ff6b7edf35d",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f8fb0f9c-fd91-4fed-aa94-b66263bd779c",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "transferDirection": "http+pushing",
    "agreementId": "urn:agreement:495117f2-9441-4d03-acc2-3b8c52549449",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-02-22T20:57:22.99903